# Potential Five-Satellite Shock Candidates

Compile candidate shocks observed while at least one MMS spacecraft was in the solar wind and THEMIS-B was probably outside Earth's magnetosphere.

The MMS requirement is deliberately one or more spacecraft, not all four. The result records the matching MMS probes and their individual solar-wind intervals.


## Selection rule

1. Read the latest SINP MSU merged shock list from ICME_list.
2. Keep each shock whose timestamp lies in a Zenodo MMS solar-wind interval for MMS1, MMS2, MMS3, or MMS4.
3. Request the local CDASWS downloader's THEMIS-B orbit product, THB_OR_SSC / XYZ_GSE.
4. Retain a potential five-satellite event when THEMIS-B is far from Earth and either on the dayside or well away from the Sun-Earth line.

This is a deliberately simple exterior proxy, not a magnetopause model: use a large geocentric distance plus either positive GSE X or a large transverse GSE Y-Z displacement. All thresholds are configurable below.


## 1. Imports


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from cdasws import CdasWs, TimeInterval
from cdasws.datarepresentation import DataRepresentation as dr


## 2. Configuration

Keep the data locations and selection thresholds together so the search is easy to retune.


In [2]:
# Resolve the repository root even when Jupyter starts in this notebook's folder.
REPO_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / '.git').exists()
)

# Input sources
ZENODO_MMS_CROSSING_URL = (
    'https://zenodo.org/records/10491878/files/'
    'solar_wind_region_list.csv?download=1'
)
ICME_LIST_DIR = Path.home() / 'Developer' / 'ICME_list'
SINP_CATALOG_PATH = ICME_LIST_DIR / 'out' / 'merged_icme_omni.csv'

# MMS and THEMIS-B selection settings
MMS_PROBES = ['mms1', 'mms2', 'mms3', 'mms4']
THEMIS_DATASET = 'THB_OR_SSC'
THEMIS_POSITION_VARIABLE = 'XYZ_GSE'
THEMIS_REQUEST_HALF_WIDTH = pd.Timedelta('5min')
THEMIS_POSITION_TOLERANCE = pd.Timedelta('5min')
# Model-free THEMIS-B exterior proxy: a distant dayside or flank position
MINIMUM_THEMIS_DAYSIDE_X_GSE_RE = 0.0
MINIMUM_THEMIS_RADIUS_RE = 20.0
MINIMUM_THEMIS_TRANSVERSE_DISTANCE_RE = 20.0

# Derived catalogue outputs live with the other downloaded data products
RESULTS_DIR = REPO_ROOT / 'Data' / 'Five Satellite Shock Candidates'
MMS_MATCHES_CSV = RESULTS_DIR / 'MMS Shock Matches.csv'
FIVE_SATELLITE_CSV = RESULTS_DIR / 'Potential Five Satellite Shocks.csv'


## 3. Load MMS solar-wind intervals

The Zenodo list supplies start, stop, probe, and region. Restrict it to the solar-wind rows and normalize all timestamps to UTC.


In [3]:
mms_regions = pd.read_csv(ZENODO_MMS_CROSSING_URL)

required_mms_columns = {'start', 'stop', 'probe', 'region'}
assert required_mms_columns.issubset(mms_regions.columns), mms_regions.columns.tolist()

mms_regions['start'] = pd.to_datetime(mms_regions['start'], utc=True)
mms_regions['stop'] = pd.to_datetime(mms_regions['stop'], utc=True)
mms_regions = mms_regions.loc[
    mms_regions['region'].eq('solar wind')
    & mms_regions['probe'].isin(MMS_PROBES)
].copy()

assert mms_regions['start'].le(mms_regions['stop']).all()
mms_regions.groupby('probe').agg(
    interval_count=('start', 'size'),
    first_start=('start', 'min'),
    last_stop=('stop', 'max'),
)


,interval_count,first_start,last_stop
probe,,,
mms1,2951,2015-10-07 12:08:00+00:00,2023-06-01 14:21:59+00:00
mms2,2974,2015-10-07 12:08:00+00:00,2023-06-01 14:21:59+00:00
mms3,3141,2015-10-07 12:08:00+00:00,2023-06-01 14:21:59+00:00
mms4,2980,2015-10-07 12:08:00+00:00,2023-06-01 14:21:59+00:00


## 4. Load the current SINP MSU shock catalogue

The current ICME_list output is the merged catalogue. Its T_shock field is the timestamp compared to the MMS intervals.


In [4]:
assert SINP_CATALOG_PATH.exists(), SINP_CATALOG_PATH

sinp_catalog = pd.read_csv(SINP_CATALOG_PATH)
assert 'T_shock' in sinp_catalog.columns, sinp_catalog.columns.tolist()

sinp_catalog['T_shock'] = pd.to_datetime(
    sinp_catalog['T_shock'],
    utc=True,
    errors='coerce',
)
catalog_columns = [column for column in sinp_catalog.columns if column != 'Unnamed: 0']
catalog_shocks = sinp_catalog.loc[sinp_catalog['T_shock'].notna()].copy()
catalog_shocks = catalog_shocks.drop_duplicates(subset='T_shock', keep='first')
catalog_shocks = catalog_shocks.sort_values('T_shock').reset_index(drop=True)

print(f'Catalog shocks with a timestamp: {len(catalog_shocks)}')
print(
    'Catalog coverage:',
    catalog_shocks['T_shock'].min(),
    'to',
    catalog_shocks['T_shock'].max(),
)
display(catalog_shocks[catalog_columns].head())


Catalog shocks with a timestamp: 1068
Catalog coverage: 1996-01-30 00:00:00+00:00 to 2026-07-03 11:17:00+00:00


,T_source,T_shock,T_start,T_end,cat,T_source_1,T_shock_1,T_start_1,T_end_1,V_icme_1,...,T_start_3,T_end_3,type_3,v_max_omni_body,v_average_omni_body,dst_min_omni_body,bx_average_omni_body,by_average_omni_body,bz_average_omni_body,temperature_omni_body
0,NaN,1996-01-30 00:00:00+00:00,1996-01-30 06:00:00,1996-01-31 07:00:00,3,NaN,NaN,NaN,NaN,NaN,...,1996-01-30 06:00:00,1996-01-31 07:00:00,EJE,573.0,525.642857,-23.0,4.221429,-1.846429,0.275000,125252.500000
1,NaN,1996-02-21 22:00:00+00:00,1996-02-22 04:00:00,1996-02-22 23:00:00,3,NaN,NaN,NaN,NaN,NaN,...,1996-02-22 04:00:00,1996-02-22 23:00:00,EJE,473.0,416.681818,-21.0,1.445455,-1.227273,-0.381818,75153.636364
2,NaN,1996-03-16 09:00:00+00:00,1996-03-17 15:00:00,1996-03-18 22:00:00,3,NaN,NaN,NaN,NaN,NaN,...,1996-03-17 15:00:00,1996-03-18 22:00:00,EJE,461.0,426.823529,-38.0,-1.111765,-1.626471,-0.700000,70933.823529
3,NaN,1996-04-03 09:00:00+00:00,1996-04-04 13:00:00,1996-04-08 12:00:00,3,NaN,NaN,NaN,NaN,NaN,...,1996-04-04 13:00:00,1996-04-08 12:00:00,EJE,411.0,344.000000,-28.0,0.360204,0.676531,-0.028571,28079.091837
4,NaN,1996-04-11 11:00:00+00:00,1996-04-12 18:00:00,1996-04-13 20:00:00,3,NaN,NaN,NaN,NaN,NaN,...,1996-04-12 18:00:00,1996-04-13 20:00:00,EJE,537.0,462.103448,-34.0,0.958621,-1.110345,-0.458621,89287.689655


## 5. Match each shock to one or more MMS solar-wind intervals

For each MMS probe, attach the latest preceding interval and test whether the shock lies before its stop. The resulting mms_probes column is the explicit answer to which MMS spacecraft matched.


In [5]:
mms_matches = catalog_shocks.copy()

for probe in MMS_PROBES:
    probe_intervals = mms_regions.loc[
        mms_regions['probe'].eq(probe),
        ['start', 'stop'],
    ].sort_values('start')

    # The crossing intervals are non-overlapping within one MMS probe.
    assert probe_intervals['start'].iloc[1:].ge(
        probe_intervals['stop'].shift().iloc[1:]
    ).all(), probe

    probe_intervals = probe_intervals.rename(
        columns={'start': f'{probe}_start', 'stop': f'{probe}_stop'}
    )
    mms_matches = pd.merge_asof(
        mms_matches.sort_values('T_shock'),
        probe_intervals,
        left_on='T_shock',
        right_on=f'{probe}_start',
        direction='backward',
    )
    mms_matches[f'{probe}_inside'] = mms_matches['T_shock'].le(
        mms_matches[f'{probe}_stop']
    )

mms_inside_columns = [f'{probe}_inside' for probe in MMS_PROBES]
mms_matches['mms_count'] = mms_matches[mms_inside_columns].sum(axis=1)
mms_matches['mms_probes'] = ''

for probe in MMS_PROBES:
    mms_matches.loc[
        mms_matches[f'{probe}_inside'],
        'mms_probes',
    ] += f'{probe}, '

mms_matches['mms_probes'] = mms_matches['mms_probes'].str.removesuffix(', ')
mms_candidates = mms_matches.loc[mms_matches['mms_count'].gt(0)].copy()

assert not mms_candidates.empty
print(f'Shocks in at least one MMS solar-wind interval: {len(mms_candidates)}')
display(mms_candidates[['T_shock', 'cat', 'mms_count', 'mms_probes']])


Shocks in at least one MMS solar-wind interval: 17


,T_shock,cat,mms_count,mms_probes
706,2017-10-21 03:00:00+00:00,2,4,"mms1, mms2, mms3, mms4"
709,2017-12-25 00:00:00+00:00,1,2,"mms1, mms2"
795,2021-12-10 13:27:00+00:00,2,3,"mms1, mms2, mms3"
798,2022-01-18 23:40:00+00:00,123,3,"mms1, mms2, mms3"
800,2022-01-24 17:09:00+00:00,2,3,"mms2, mms3, mms4"
801,2022-02-01 22:20:00+00:00,12,1,mms3
805,2022-03-10 18:37:00+00:00,2,4,"mms1, mms2, mms3, mms4"
808,2022-03-19 20:57:00+00:00,2,4,"mms1, mms2, mms3, mms4"
811,2022-04-08 03:39:00+00:00,2,4,"mms1, mms2, mms3, mms4"
812,2022-04-08 19:00:00+00:00,3,4,"mms1, mms2, mms3, mms4"


## 6. Download closest THEMIS-B positions with CDASWS

This copies the approach used by CDASWS Downloader.ipynb: CdasWs, TimeInterval, and SPACEPY representation. THB_OR_SSC supplies XYZ_GSE in Earth radii, so no unit conversion is needed.


In [6]:
cdas = CdasWs()
themis_position_frames = []

for shock_time in mms_candidates['T_shock']:
    request_interval = TimeInterval(
        (shock_time - THEMIS_REQUEST_HALF_WIDTH).to_pydatetime(),
        (shock_time + THEMIS_REQUEST_HALF_WIDTH).to_pydatetime(),
    )
    _, themis_state = cdas.get_data(
        THEMIS_DATASET,
        [THEMIS_POSITION_VARIABLE],
        request_interval,
        dataRepresentation=dr.SPACEPY,
    )

    position_frame = pd.DataFrame(
        themis_state[THEMIS_POSITION_VARIABLE],
        index=pd.to_datetime(themis_state['Epoch'], utc=True),
        columns=['thb_x_gse_re', 'thb_y_gse_re', 'thb_z_gse_re'],
    )

    # Remove the CDAS fill value before the nearest-time match.
    position_frame = position_frame.mask(position_frame.abs().gt(1.0e4), np.nan)
    assert position_frame.notna().any(axis=None), shock_time
    themis_position_frames.append(position_frame)

themis_positions = pd.concat(themis_position_frames).sort_index()
themis_positions = themis_positions.loc[
    ~themis_positions.index.duplicated(keep='first')
]
themis_positions = themis_positions.rename_axis('themis_time').reset_index()

display(themis_positions.head())


/opt/homebrew/Caskroom/miniconda/base/envs/icme3.12-metal/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


,themis_time,thb_x_gse_re,thb_y_gse_re,thb_z_gse_re
0,2017-10-21 02:55:00+00:00,57.379339,17.210506,5.021151
1,2017-10-21 02:56:00+00:00,57.377459,17.221277,5.021324
2,2017-10-21 02:57:00+00:00,57.375643,17.232177,5.021562
3,2017-10-21 02:58:00+00:00,57.373826,17.243072,5.021813
4,2017-10-21 02:59:00+00:00,57.371946,17.253843,5.021986


## 7. Apply the THEMIS-B outside-magnetosphere proxy

A five-satellite candidate must have a position within the stated tolerance, be at least the configured distance from Earth, and be either on the dayside or far from the Sun-Earth line in the GSE Y-Z plane. This allows a flank position with large absolute GSE Y to qualify without invoking a magnetosphere model.


In [7]:
candidate_positions = pd.merge_asof(
    mms_candidates.sort_values('T_shock'),
    themis_positions.sort_values('themis_time'),
    left_on='T_shock',
    right_on='themis_time',
    direction='nearest',
    tolerance=THEMIS_POSITION_TOLERANCE,
)

assert candidate_positions['thb_x_gse_re'].notna().all()
candidate_positions['themis_time_offset_s'] = (
    candidate_positions['themis_time'] - candidate_positions['T_shock']
).dt.total_seconds()
candidate_positions['thb_radius_re'] = np.sqrt(
    candidate_positions['thb_x_gse_re'] ** 2
    + candidate_positions['thb_y_gse_re'] ** 2
    + candidate_positions['thb_z_gse_re'] ** 2
)
candidate_positions['thb_transverse_distance_re'] = np.sqrt(
    candidate_positions['thb_y_gse_re'] ** 2
    + candidate_positions['thb_z_gse_re'] ** 2
)
candidate_positions['themis_b_probably_outside_magnetosphere'] = (
    candidate_positions['thb_radius_re'].ge(MINIMUM_THEMIS_RADIUS_RE)
    & (
        candidate_positions['thb_x_gse_re'].gt(MINIMUM_THEMIS_DAYSIDE_X_GSE_RE)
        | candidate_positions['thb_transverse_distance_re'].ge(
            MINIMUM_THEMIS_TRANSVERSE_DISTANCE_RE
        )
    )
)

display(
    candidate_positions[
        [
            'T_shock',
            'cat',
            'mms_probes',
            'thb_x_gse_re',
            'thb_y_gse_re',
            'thb_z_gse_re',
            'themis_time_offset_s',
            'thb_radius_re',
            'thb_transverse_distance_re',
            'themis_b_probably_outside_magnetosphere',
        ]
    ]
)


,T_shock,cat,mms_probes,thb_x_gse_re,thb_y_gse_re,thb_z_gse_re,themis_time_offset_s,thb_radius_re,thb_transverse_distance_re,themis_b_probably_outside_magnetosphere
0,2017-10-21 03:00:00+00:00,2,"mms1, mms2, mms3, mms4",57.370130,17.264741,5.022223,0.0,60.121758,17.980378,True
1,2017-12-25 00:00:00+00:00,1,"mms1, mms2",18.457461,61.105488,-2.901006,0.0,63.898156,61.174313,True
2,2021-12-10 13:27:00+00:00,2,"mms1, mms2, mms3",6.852132,59.423575,-5.390406,0.0,60.059716,59.667560,True
3,2022-01-18 23:40:00+00:00,123,"mms1, mms2, mms3",-60.356793,-12.348266,5.186300,0.0,61.824912,13.393184,False
4,2022-01-24 17:09:00+00:00,2,"mms2, mms3, mms4",-9.132610,-56.236096,3.037488,0.0,57.053741,56.318068,True
5,2022-02-01 22:20:00+00:00,12,mms3,58.593589,11.150752,-5.005396,0.0,59.854840,12.222653,True
6,2022-03-10 18:37:00+00:00,2,"mms1, mms2, mms3, mms4",-2.654005,61.991173,2.551814,0.0,62.100411,62.043672,True
7,2022-03-19 20:57:00+00:00,2,"mms1, mms2, mms3, mms4",-53.004290,-21.517860,2.810775,0.0,57.274545,21.700662,True
8,2022-04-08 03:39:00+00:00,2,"mms1, mms2, mms3, mms4",15.342214,61.128801,3.986385,0.0,63.150654,61.258645,True
9,2022-04-08 19:00:00+00:00,3,"mms1, mms2, mms3, mms4",6.488042,60.933370,4.188164,0.0,61.420770,61.077134,True


## 8. Rank candidates by Dst

dst_min_omni_body is the minimum Dst during the catalogue's OMNI ICME interval. More-negative values rank first; this is a storm-severity screen, not a measure of the shock jump itself.


In [ ]:
dst_ranked_candidates = candidate_positions.loc[
    candidate_positions['themis_b_probably_outside_magnetosphere']
].sort_values('dst_min_omni_body', ascending=True, na_position='last')

display(
    dst_ranked_candidates[
        [
            'T_shock',
            'cat',
            'mms_count',
            'mms_probes',
            'dst_min_omni_body',
            'thb_x_gse_re',
            'thb_y_gse_re',
            'thb_z_gse_re',
        ]
    ]
)


## 9. Review before downloading event data

- Check mms_probes and the probe-specific start/stop columns before deciding which MMS data to download.
- Check themis_time_offset_s is acceptably small for every retained event.
- Tune MINIMUM_THEMIS_DAYSIDE_X_GSE_RE, MINIMUM_THEMIS_RADIUS_RE, or MINIMUM_THEMIS_TRANSVERSE_DISTANCE_RE to make the no-magnetosphere proxy more or less conservative.
- Use the existing CDASWS Downloader notebook to retrieve plasma and magnetic-field data for the final rows.
